# CFPB Consumer Complaints - inspeção inicial dos dados

Este notebook verifica a integridade e a estrutura do arquivo integral do **Consumer Complaint Database (CFPB)** antes de qualquer definição de tarefa ou modelagem.

Objetivos desta etapa:

- confirmar os arquivos e o ambiente de execução;
- medir volume, período, cobertura das narrativas e cardinalidade dos rótulos;
- observar o desbalanceamento de `Product` e `Issue`;
- inspecionar textos de forma limitada e auditável;
- demonstrar a fronteira entre Polars e scikit-learn sem tratar o smoke test como avaliação de modelo.

> O CSV bruto tem aproximadamente 9 GB. As células marcadas como **varredura completa** usam execução lazy/streaming, mas ainda precisam ler o arquivo inteiro.

In [ ]:
from pathlib import Path
import json
import platform
import subprocess
import sys

import duckdb
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import polars as pl
import pyarrow as pa
import sklearn

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "dataset").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "dataset" / "raw"
RAW_CSV = RAW_DIR / "complaints.csv"
RAW_ZIP = RAW_DIR / "complaints.csv.zip"
PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed"
PARQUET_PATH = PROCESSED_DIR / "complaints.parquet"
DUCKDB_TEMP_DIR = PROJECT_ROOT / "temp" / "duckdb"

print(f"Python: {platform.python_version()}")
print(f"Polars: {pl.__version__}")
print(f"DuckDB: {duckdb.__version__}")
print(f"PyArrow: {pa.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"Projeto: {PROJECT_ROOT}")

for path in (RAW_ZIP, RAW_CSV):
    assert path.exists(), f"Arquivo não encontrado: {path}"
    print(f"{path.name}: {path.stat().st_size / 1_000_000_000:.3f} GB")

DUCKDB_TEMP_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_SQL = PARQUET_PATH.as_posix().replace("'", "''")
DUCKDB_RUNNER = f'''
import duckdb
import json
import sys
con = duckdb.connect()
con.execute(\"SET memory_limit = '2GB'\")
con.execute(\"SET threads = 2\")
con.execute(\"SET temp_directory = '{DUCKDB_TEMP_DIR.as_posix()}'\")
con.execute(\"SET preserve_insertion_order = false\")
result = con.execute(sys.stdin.read())
columns = [item[0] for item in result.description]
rows = result.fetchall()
print(json.dumps({{\"columns\": columns, \"rows\": rows}}, default=str))
'''

def duckdb_query(sql: str) -> pl.DataFrame:
    completed = subprocess.run(
        [sys.executable, "-X", "utf8", "-c", DUCKDB_RUNNER],
        input=sql.encode("utf-8"),
        capture_output=True,
        check=True,
    )
    stdout = completed.stdout.decode("utf-8", errors="replace")
    payload_line = next(line for line in reversed(stdout.splitlines()) if line.strip())
    payload = json.loads(payload_line)
    return pl.DataFrame(payload["rows"], schema=payload["columns"], orient="row")

## Papel do Polars no pipeline de NLP

Polars será a camada de dados: leitura, limpeza, filtros, agregações, criação dos splits e persistência em Parquet. Ele não substitui as representações usadas pelos modelos.

Na fronteira de treinamento:

- `TfidfVectorizer` recebe uma sequência de strings e produz uma matriz esparsa SciPy;
- estimadores scikit-learn consomem essa matriz esparsa ou arrays NumPy;
- tokenizadores de transformers recebem lotes de strings;
- somente o recorte necessário é convertido com `.to_list()` ou `.to_numpy()`.

Não converteremos o corpus completo para pandas apenas para treinar um modelo.

In [ ]:
TEXT_COL = "Consumer complaint narrative"
DATE_COL = "Date received"
PRODUCT_COL = "Product"
ISSUE_COL = "Issue"

lf = pl.scan_csv(
    RAW_CSV,
    schema_overrides={
        DATE_COL: pl.String,
        PRODUCT_COL: pl.String,
        ISSUE_COL: pl.String,
        TEXT_COL: pl.String,
    },
)

schema = lf.collect_schema()
print(f"Colunas: {len(schema)}")
schema

## Camada analítica Parquet - conversão única

A primeira execução converte o CSV integral para Parquet com compressão Zstandard e row groups de 250 mil linhas. O CSV e o ZIP permanecem imutáveis em `dataset/raw`. Execuções posteriores reutilizam o Parquet.

> Esta é uma **varredura completa** do CSV, mas ocorre apenas uma vez. Um arquivo temporário recebe a escrita e só é promovido ao nome final depois do fechamento bem-sucedido.

In [ ]:
%%time
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
temporary_parquet = PROCESSED_DIR / "_complaints.parquet.incomplete"

if not PARQUET_PATH.exists():
    if temporary_parquet.exists():
        temporary_parquet.unlink()
    lf.sink_parquet(
        temporary_parquet,
        compression="zstd",
        compression_level=3,
        row_group_size=250_000,
        maintain_order=True,
        mkdir=True,
    )
    temporary_parquet.replace(PARQUET_PATH)

print(f"Parquet: {PARQUET_PATH.stat().st_size / 1_000_000_000:.3f} GB")
lf = pl.scan_parquet(PARQUET_PATH)

## Amostra de inspeção

Esta leitura limitada serve apenas para observar o formato. Como o arquivo não está ordenado aleatoriamente, proporções calculadas nesta amostra não devem ser interpretadas como estimativas do corpus completo.

In [ ]:
sample = pl.read_parquet(PARQUET_PATH, n_rows=25_000)

sample.filter(pl.col(TEXT_COL).is_not_null()).select(
    DATE_COL,
    PRODUCT_COL,
    ISSUE_COL,
    pl.col(TEXT_COL).str.slice(0, 300).alias("Narrativa (até 300 caracteres)"),
).head(10)

## Perfil global com DuckDB out-of-core

A célula abaixo consulta o Parquet com limite de memória de 2 GB, dois threads e spill para o disco D:. As operações são separadas para controlar o pico de memória. Contagens e cardinalidades são exatas; os quantis de comprimento usam `approx_quantile`.

In [ ]:
%%time
volume = duckdb_query(f"""
    SELECT
        count(*)::BIGINT,
        count(*) FILTER (
            WHERE trim(coalesce(\"Consumer complaint narrative\", '')) <> ''
        )::BIGINT,
        min(\"Date received\"),
        max(\"Date received\")
    FROM read_parquet('{PARQUET_SQL}')
""").row(0)
cardinality = duckdb_query(f"""
    SELECT count(DISTINCT \"Product\"), count(DISTINCT \"Issue\")
    FROM read_parquet('{PARQUET_SQL}')
""").row(0)
length_quantiles = duckdb_query(f"""
    SELECT
        approx_quantile(length(narrative), 0.50),
        approx_quantile(length(narrative), 0.90),
        approx_quantile(length(narrative), 0.99)
    FROM (
        SELECT trim(coalesce(\"Consumer complaint narrative\", '')) AS narrative
        FROM read_parquet('{PARQUET_SQL}')
    )
    WHERE narrative <> ''
""").row(0)

profile = pl.DataFrame({
    "total_registros": [volume[0]],
    "com_narrativa": [volume[1]],
    "data_min": [volume[2]],
    "data_max": [volume[3]],
    "produtos_distintos": [cardinality[0]],
    "issues_distintos": [cardinality[1]],
    "narrativa_mediana_chars": [length_quantiles[0]],
    "narrativa_p90_chars": [length_quantiles[1]],
    "narrativa_p99_chars": [length_quantiles[2]],
}).with_columns(
    (100 * pl.col("com_narrativa") / pl.col("total_registros"))
    .round(2)
    .alias("percentual_com_narrativa")
)

profile

## Distribuição dos rótulos - varredura completa

Esta etapa executa agregações exatas para `Product` e `Issue`. O desbalanceamento e as mudanças históricas de taxonomia precisam ser considerados antes de definir as classes do problema.

In [ ]:
%%time
top_products = duckdb_query(f"""
    SELECT \"Product\", count(*)::BIGINT AS registros
    FROM read_parquet('{PARQUET_SQL}')
    GROUP BY 1 ORDER BY 2 DESC LIMIT 20
""")
top_issues = duckdb_query(f"""
    SELECT \"Issue\", count(*)::BIGINT AS registros
    FROM read_parquet('{PARQUET_SQL}')
    GROUP BY 1 ORDER BY 2 DESC LIMIT 20
""")
display(top_products)
display(top_issues)

In [ ]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Top 12 por produto", "Top 12 por issue"),
    horizontal_spacing=0.28,
)

for column, frame in enumerate((top_products.head(12), top_issues.head(12)), start=1):
    plot_frame = frame.sort("registros")
    labels = [str(value)[:65] for value in plot_frame[:, 0].to_list()]
    fig.add_trace(
        go.Bar(
            x=plot_frame["registros"].to_list(),
            y=labels,
            orientation="h",
            showlegend=False,
        ),
        row=1,
        col=column,
    )
    fig.update_xaxes(title_text="Registros", row=1, col=column)

fig.update_layout(height=650, width=1250, margin=dict(l=20, r=20, t=70, b=40))
fig.show()

## Smoke test: Polars → TF-IDF → scikit-learn

O objetivo desta célula é verificar interoperabilidade, não estimar desempenho. Ela usa apenas a amostra inicial, sem split temporal, consolidação de taxonomia, deduplicação ou auditoria de vazamento. Nenhuma métrica gerada aqui deve ser publicada como resultado do projeto.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

model_sample = sample.filter(
    pl.col(TEXT_COL).is_not_null(),
    pl.col(PRODUCT_COL).is_not_null(),
).with_columns(
    pl.col(TEXT_COL).str.strip_chars(),
).filter(pl.col(TEXT_COL).ne(""))

eligible_products = (
    model_sample.group_by(PRODUCT_COL)
    .len(name="n")
    .filter(pl.col("n") >= 100)
    .sort("n", descending=True)
    .head(5)[PRODUCT_COL]
).to_list()
model_sample = model_sample.filter(pl.col(PRODUCT_COL).is_in(eligible_products))

X_train, X_test, y_train, y_test = train_test_split(
    model_sample[TEXT_COL].to_list(),
    model_sample[PRODUCT_COL].to_list(),
    test_size=0.25,
    random_state=42,
    stratify=model_sample[PRODUCT_COL].to_list(),
)

vectorizer = TfidfVectorizer(
    min_df=3,
    max_features=20_000,
    ngram_range=(1, 2),
    sublinear_tf=True,
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=500, class_weight="balanced")
model.fit(X_train_tfidf, y_train)
pred = model.predict(X_test_tfidf)

print(f"Treino: {len(X_train):,} textos")
print(f"Matriz TF-IDF: {X_train_tfidf.shape}, tipo={type(X_train_tfidf).__name__}")
print(classification_report(y_test, pred, zero_division=0))

## Próximas decisões após a inspeção

1. Definir uma taxonomia consolidada que não misture simples renomeações históricas com classes distintas.
2. Quantificar duplicatas e textos padronizados/repetitivos antes do split.
3. Escolher uma janela temporal e congelar treino, validação e teste por data.
4. Definir a tarefa primária: produto, issue hierárquico ou roteamento com abstention.
5. Converter o recorte analítico aprovado para Parquet, preservando o CSV original como fonte imutável.